In [5]:
import dotenv
%load_ext dotenv
%dotenv

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [6]:
import nest_asyncio
nest_asyncio.apply()

### Create the knowledge base for document collection

In [7]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.readers.file import PDFReader

splitter = SentenceSplitter(chunk_size=1024)

documents = SimpleDirectoryReader(input_dir="./datasets", file_extractor={".pdf": PDFReader()}).load_data()
nodes = splitter.get_nodes_from_documents(documents)
nodes

[TextNode(id_='6518cbc1-9b7f-4869-9220-8373ce12ad01', embedding=None, metadata={'page_label': '1', 'file_name': 'longlora_efficient_fine_tuning.pdf', 'file_path': 'c:\\Users\\ahcha\\OneDrive\\桌面\\AgenticRAG\\Basics\\datasets\\longlora_efficient_fine_tuning.pdf', 'file_type': 'application/pdf', 'file_size': 1168720, 'creation_date': '2026-06-24', 'last_modified_date': '2026-06-24'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='16cb7870-0d4b-4dff-8cb2-e3fa433ff897', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'page_label': '1', 'file_name': 'longlora_efficient_fine_tuning.pdf', 'file_path': 'c:\\Users\\ahcha\\OneDrive\\桌面\\AgenticRAG\\Basics\\datasets\\longlora_efficient_fine_tuning.pdf', 'file_type': 

In [8]:
import pandas as pd

df = pd.DataFrame([{"text": node.text} for node in nodes])
df.head(10)

,text
0,Published as a conference paper at ICLR 2024\n...
1,Published as a conference paper at ICLR 2024\n...
2,"LoRA (Hu et al., 2022) uses\nlow-rank weight u..."
3,Published as a conference paper at ICLR 2024\n...
4,They alleviate the quadratic complexity of\nth...
5,Published as a conference paper at ICLR 2024\n...
6,A\nfinal normalization is conducted after all ...
7,Published as a conference paper at ICLR 2024\n...
8,"However, most of them are not suitable for lon..."
9,Published as a conference paper at ICLR 2024\n...


In [14]:
import os
import giskard

giskard.llm.set_llm_model("anthropic/claude-haiku-4-5", api_key=os.environ["ANTHROPIC_API_KEY"])

In [15]:
ollama_local_server = "http://localhost:11434"

giskard.llm.set_embedding_model("ollama/nomic-embed-text-v2-moe", api_base=ollama_local_server)

In [16]:
from giskard.rag import KnowledgeBase

knowledge_base = KnowledgeBase(df)

### Generate the test set

In [17]:
from giskard.rag import generate_testset

testset = generate_testset(
    knowledge_base,
    num_questions=60,
    language='en',
    agent_description="A chatbot answering questions to a collection of documents related to machine learning techniques",
)

2026-06-30 00:50:31,362 - INFO - Finding topics in the knowledge base.
2026-06-30 00:50:40,389 - INFO - Wrapper: Completed Call, calling success_handler
2026-06-30 00:50:43,258 - INFO - Wrapper: Completed Call, calling success_handler
c:\Users\ahcha\OneDrive\桌面\AgenticRAG\Basics\.venv312\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
2026-06-30 00:50:49,380 - INFO - 
LiteLLM completion() model= claude-haiku-4-5; provider = anthropic
2026-06-30 00:50:50,330 - INFO - Wrapper: Completed Call, calling success_handler
2026-06-30 00:50:50,331 - INFO - 
LiteLLM completion() model= claude-haiku-4-5; provider = anthropic
2026-06-30 00:50:51,042 - INFO - Wrapper: Completed Call, calling success_handler
2026-06-30 00:50:51,044 - INFO - 
LiteLLM completion() model= claude-haiku-4-5; provider = anthropic
2026-06-30 00:50:51,858 - INFO - Wr

In [18]:
testset_df = testset.to_pandas()

for index, row in enumerate(testset_df.head(3).iterrows()):
    print(f"Question {index + 1}: {row[1]['question']}")
    print(f"Reference answer: {row[1]['reference_answer']}")
    print("Reference context:")
    print(row[1]['reference_context'])
    print("******************", end="\n\n")

Question 1: How does LoRA perform compared to Fine-Tune on the MNLI dataset in low-data regimes?
Reference answer: LoRA achieves better performance than fine-tuning on both MNLI-100 and MNLI-Full, and comparable results on MNLI-1k and MNLI-10K considering the (±0.3) variance due to random seeds.
Reference context:
Document 68: Method WebNLG
BLEU↑ MET↑ TER↓
U S A U S A U S A
GPT-2 Medium
Fine-Tune (354M) 27.7 64.2 46.5 .30 .45 .38 .76 .33 .53
AdapterL (0.37M) 45.1 54.5 50.2 .36 .39 .38 .46 .40 .43
AdapterL (11M) 48.3 60.4 54.9 .38 .43 .41 .45 .35 .39
FTTop2 (24M) 18.9 53.6 36.0 .23 .38 .31 .99 .49 .72
Preﬁx (0.35M) 45.6 62.9 55.1 .38 .44 .41 .49 .35 .40
LoRA (0.35M) 46.7±.4 62.1±.2 55.3±.2 .38 .44 .41 .46 .33 .39
GPT-2 Large
Fine-Tune (774M) 43.1 65.3 55.5 .38 .46 .42 .53 .33 .42
AdapterL (0.88M) 49.8±.0 61.1±.0 56.0±.0 .38 .43 .41 .44 .35 .39
AdapterL (23M) 49.2±.1 64.7±.2 57.7±.1 .39 .46 .43 .46 .33 .39
Preﬁx (0.77M) 47.7 63.4 56.3 .39 .45 .42 .48 .34 .40
LoRA (0.77M) 48.4±.3 64.0±.3 

In [19]:
testset.save("./testsets/my_testset.jsonl")

### Build the agentic RAG

In [29]:
from agent import create_tools_for_papers, create_tool_retriever, create_agent, run_query
from pathlib import Path
from llama_index.llms.anthropic import Anthropic

papers = [str(p) for p in Path("./datasets").iterdir() if p.is_file()]

all_tools = await create_tools_for_papers(papers)
obj_retriever = create_tool_retriever(all_tools, similarity_top_k=5)

llm = Anthropic(model="claude-haiku-4-5")
agent = create_agent(obj_retriever, llm, verbose=False)

Creating tools for datasets\longlora_efficient_fine_tuning.pdf
Creating tools for datasets\lora_paper.pdf


In [22]:
response = await run_query(agent, "What is Lora?")

In [23]:
print(str(response))

## LoRA: Low-Rank Adaptation of Large Language Models

**LoRA** is a parameter-efficient adaptation method designed to fine-tune large language models while dramatically reducing computational and storage requirements.

### Core Concept

Instead of retraining all model parameters, LoRA:
- **Freezes** the pre-trained model weights
- **Injects** trainable rank decomposition matrices into each layer of the Transformer architecture
- Uses a low-rank decomposition to represent weight updates

### How It Works

For a pre-trained weight matrix W₀, LoRA constrains its update using:

**W₀ + ΔW = W₀ + BA**

Where:
- **B** and **A** are trainable matrices with rank r ≪ min(d,k)
- **W₀** remains frozen during training
- The forward pass becomes: **h = W₀x + BAx**

### Key Advantages

1. **Massive Parameter Reduction**: Reduces trainable parameters by up to 10,000× compared to full fine-tuning (e.g., from 175B to 35MB for GPT-3)

2. **No Inference Latency**: The trainable matrices can be merged wit

### Evaluate the agentic RAG on the test set

In [30]:
import asyncio

# Wrap your RAG model
def get_answer_fn(question: str, history=None) -> str:
    """A function representing your RAG agent."""
    # Format appropriately the history for your RAG agent
    messages = history if history else []
    messages.append({"role": "user", "content": question})

    if len(messages) > 1:  # there's actual history
        user_msg = "\n".join(f"{m['role'].capitalize()}: {m['content']}" for m in messages)
    else:
        user_msg = question

    # Get the answer
    answer = asyncio.get_event_loop().run_until_complete(run_query(agent, user_msg))
    return str(answer)

In [31]:
from giskard.rag import evaluate

report = evaluate(get_answer_fn, testset=testset, knowledge_base=knowledge_base)

CorrectnessMetric evaluation:   0%|          | 0/58 [00:00<?, ?it/s]2026-07-01 00:46:52,254 - INFO - 
LiteLLM completion() model= claude-haiku-4-5; provider = anthropic
2026-07-01 00:46:54,211 - INFO - Wrapper: Completed Call, calling success_handler
CorrectnessMetric evaluation:   2%|▏         | 1/58 [00:01<01:51,  1.96s/it]2026-07-01 00:46:54,213 - INFO - 
LiteLLM completion() model= claude-haiku-4-5; provider = anthropic
2026-07-01 00:46:55,393 - INFO - Wrapper: Completed Call, calling success_handler
CorrectnessMetric evaluation:   3%|▎         | 2/58 [00:03<01:24,  1.50s/it]2026-07-01 00:46:55,395 - INFO - 
LiteLLM completion() model= claude-haiku-4-5; provider = anthropic
2026-07-01 00:46:58,666 - INFO - Wrapper: Completed Call, calling success_handler
CorrectnessMetric evaluation:   5%|▌         | 3/58 [00:06<02:07,  2.31s/it]2026-07-01 00:46:58,668 - INFO - 
LiteLLM completion() model= claude-haiku-4-5; provider = anthropic
2026-07-01 00:47:00,518 - INFO - Wrapper: Completed Ca

In [32]:
display(report)

Loading BokehJS ...

In [33]:
report.correctness_by_question_type()

,correctness
question_type,
complex,0.600000
conversational,0.555556
distracting element,0.700000
double,0.666667
simple,0.600000
situational,0.900000
